In [117]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import nltk
from nltk.corpus import stopwords
import re
import jieba

In [118]:
movies = {
    ('十二怒汉', '12 Angry Men'): [['西德尼·吕美特', 'Sidney Lumet'], ['亨利·方达', 'Henry Fonda', '李·J·科布', 'Lee J. Cobb', '埃德·贝格利', 'Ed Begley']],
    ('2001太空漫游', '2001: A Space Odyssey'): [['斯坦利·库布里克', 'Stanley Kubrick'], ['凯尔·杜拉', 'Keir Dullea', '加里·洛克伍德', 'Gary Lockwood', '威廉·西尔维斯特', 'William Sylvester']],
    ('美丽心灵', 'A Beautiful Mind'): [['朗·霍华德', 'Ron Howard'], ['罗素·克劳', 'Russell Crowe', '詹妮弗·康纳利', 'Jennifer Connelly', '艾德·哈里斯', 'Ed Harris']],
    ('阿凡达', 'Avatar'): [['詹姆斯·卡梅隆', 'James Cameron'], ['萨姆·沃辛顿', 'Sam Worthington', '佐伊·索尔达娜', 'Zoe Saldana', '西格妮·韦弗', 'Sigourney Weaver']],
    ('黑天鹅', 'Black Swan'): [['达伦·阿罗诺夫斯基', 'Darren Aronofsky'], ['娜塔莉·波特曼', 'Natalie Portman', '米拉·库尼斯', 'Mila Kunis', '文森特·卡塞尔', 'Vincent Cassel']],
    ('波西米亚狂想曲', 'Bohemian Rhapsody'): [['布赖恩·辛格', 'Bryan Singer'], ['拉米·马雷克', 'Rami Malek', '露西·宝通', 'Lucy Boynton', '格威利姆·李', 'Gwilym Lee']],
    ('逍遥法外', 'Catch Me If You Can'): [['史蒂文·斯皮尔伯格', 'Steven Spielberg'], ['莱昂纳多·迪卡普里奥', 'Leonardo DiCaprio', '汤姆·汉克斯', 'Tom Hanks', '克里斯托弗·沃肯', 'Christopher Walken']],
    ('寻梦环游记', 'Coco'): [['李·昂克里奇', 'Lee Unkrich', '阿德里安·莫利纳', 'Adrian Molina'], ['安东尼·冈萨雷斯', 'Anthony Gonzalez', '盖尔·加西亚·贝纳尔', 'Gael García Bernal', '本杰明·布拉特', 'Benjamin Bratt']],
    ('被解救的姜戈', 'Django Unchained'): [['昆汀·塔伦蒂诺', 'Quentin Tarantino'], ['杰米·福克斯', 'Jamie Foxx', '克里斯托弗·瓦尔茨', 'Christoph Waltz', '莱昂纳多·迪卡普里奥', 'Leonardo DiCaprio']],
    ('搏击俱乐部', 'Fight Club'): [['大卫·芬奇', 'David Fincher'], ['布拉德·皮特', 'Brad Pitt', '爱德华·诺顿', 'Edward Norton', '海伦娜·邦汉·卡特', 'Helena Bonham Carter']],
    ('阿甘正传', 'Forrest Gump'): [['罗伯特·泽米吉斯', 'Robert Zemeckis'], ['汤姆·汉克斯', 'Tom Hanks', '罗宾·怀特', 'Robin Wright', '加里·西尼斯', 'Gary Sinise']],
    ('冰雪奇缘', 'Frozen'): [['克里斯·巴克', 'Chris Buck', '詹妮弗·李', 'Jennifer Lee'], ['克里斯汀·贝尔', 'Kristen Bell', '伊迪娜·门泽尔', 'Idina Menzel', '乔纳森·格罗夫', 'Jonathan Groff']],
    ('消失的爱人', 'Gone Girl'): [['大卫·芬奇', 'David Fincher'], ['本·阿弗莱克', 'Ben Affleck', '罗莎蒙德·派克', 'Rosamund Pike', '尼尔·帕特里克·哈里斯', 'Neil Patrick Harris']],
    ('心灵捕手', 'Good Will Hunting'): [['格斯·范·桑特', 'Gus Van Sant'], ['马特·达蒙', 'Matt Damon', '罗宾·威廉姆斯', 'Robin Williams', '本·阿弗莱克', 'Ben Affleck']],
    ('绿皮书', 'Green Book'): [['彼得·法拉利', 'Peter Farrelly'], ['维果·莫滕森', 'Viggo Mortensen', '马赫沙拉·阿里', 'Mahershala Ali', '琳达·卡德里尼', 'Linda Cardellini']],
    ('血战钢锯岭', 'Hacksaw Ridge'): [['梅尔·吉布森', 'Mel Gibson'], ['安德鲁·加菲尔德', 'Andrew Garfield', '萨姆·沃辛顿', 'Sam Worthington', '卢克·布雷西', 'Luke Bracey']],
    ('哈利·波特与魔法石', "Harry Potter and the Sorcerer's Stone"): [['克里斯·哥伦布', 'Chris Columbus'], ['丹尼尔·雷德克里夫', 'Daniel Radcliffe', '鲁伯特·格林特', 'Rupert Grint', '艾玛·沃森', 'Emma Watson']],
    ('驯龙高手', 'How to Train Your Dragon'): [['迪恩·德布洛伊斯', 'Dean DeBlois', '克里斯·桑德斯', 'Chris Sanders'], ['杰伊·巴鲁切尔', 'Jay Baruchel', '杰拉德·巴特勒', 'Gerard Butler', '克雷格·弗格森', 'Craig Ferguson']],
    ('盗梦空间', 'Inception'): [['克里斯托弗·诺兰', 'Christopher Nolan'], ['莱昂纳多·迪卡普里奥', 'Leonardo DiCaprio', '玛丽昂·歌迪亚', 'Marion Cotillard', '汤姆·哈迪', 'Tom Hardy']],
    ('无耻混蛋', 'Inglourious Basterds'): [['昆汀·塔伦蒂诺', 'Quentin Tarantino'], ['布拉德·皮特', 'Brad Pitt', '克里斯托弗·瓦尔茨', 'Christoph Waltz', '梅拉尼·劳伦', 'Mélanie Laurent']],
    ('头脑特工队', 'Inside Out'): [['皮特·道格特', 'Pete Docter', '罗尼·德尔·卡门', 'Ronnie Del Carmen'], ['艾米·波勒', 'Amy Poehler', '菲利斯·史密斯', 'Phyllis Smith', '理查德·金德', 'Richard Kind']],
    ('星际穿越', 'Interstellar'): [['克里斯托弗·诺兰', 'Christopher Nolan'], ['马修·麦康纳', 'Matthew McConaughey', '安妮·海瑟薇', 'Anne Hathaway', '杰西卡·查斯坦', 'Jessica Chastain']],
    ('小丑', 'Joker'): [['托德·菲利普斯', 'Todd Phillips'], ['华金·菲尼克斯', 'Joaquin Phoenix', '罗伯特·德尼罗', 'Robert De Niro', '扎齐·贝茨', 'Zazie Beetz']],
    ('爱乐之城', 'La La Land'): [['达米恩·查泽雷', 'Damien Chazelle'], ['瑞恩·高斯林', 'Ryan Gosling', '艾玛·斯通', 'Emma Stone', '约翰·传奇', 'John Legend']],
    ('美丽人生', 'Life Is Beautiful'): [['罗伯托·贝尼尼', 'Roberto Benigni'], ['罗伯托·贝尼尼', 'Roberto Benigni', '尼科莱塔·布拉斯基', 'Nicoletta Braschi', '乔治·坎塔里尼', 'Giorgio Cantarini']],
    ('少年派的奇幻漂流', 'Life of Pi'): [['李安', 'Ang Lee'], ['苏拉·沙玛', 'Suraj Sharma', '伊尔凡·汗', 'Irrfan Khan', '拉夫·斯波', 'Rafe Spall']],
    ('两杆大烟枪', 'Lock, Stock and Two Smoking Barrels'): [['盖·里奇', 'Guy Ritchie'], ['杰森·弗莱明', 'Jason Flemyng', '德克斯特·弗莱彻', 'Dexter Fletcher', '尼克·莫兰', 'Nick Moran']],
    ('这个杀手不太冷', 'Léon: The Professional'): [['吕克·贝松', 'Luc Besson'], ['让·雷诺', 'Jean Reno', '娜塔莉·波特曼', 'Natalie Portman', '加里·奥德曼', 'Gary Oldman']],
    ('疯狂的麦克斯：狂暴之路', 'Mad Max: Fury Road'): [['乔治·米勒', 'George Miller'], ['汤姆·哈迪', 'Tom Hardy', '查理兹·塞隆', 'Charlize Theron', '尼古拉斯·霍尔特', 'Nicholas Hoult']],
    ('记忆碎片', 'Memento'): [['克里斯托弗·诺兰', 'Christopher Nolan'], ['盖·皮尔斯', 'Guy Pearce', '凯瑞-安·莫斯', 'Carrie-Anne Moss', '乔·潘托里亚诺', 'Joe Pantoliano']],
    ('怪兽电力公司', 'Monsters, Inc.'): [['皮特·道格特', 'Pete Docter', '大卫·西尔弗曼', 'David Silverman', '李·昂克里奇', 'Lee Unkrich'], ['约翰·古德曼', 'John Goodman', '比利·克里斯托', 'Billy Crystal', '玛丽·吉布斯', 'Mary Gibbs']],
    ('飞越疯人院', "One Flew Over the Cuckoo's Nest"): [['米洛斯·福尔曼', 'Miloš Forman'], ['杰克·尼科尔森', 'Jack Nicholson', '路易丝·弗莱彻', 'Louise Fletcher', '丹尼·德维托', 'Danny DeVito']],
    ('寄生虫', 'Parasite'): [['奉俊昊', 'Bong Joon-ho'], ['宋康昊', 'Song Kang-ho', '李善均', 'Lee Sun-kyun', '赵汝贞', 'Cho Yeo-jeong']],
    ('加勒比海盗：黑珍珠号的诅咒', 'Pirates of the Caribbean: The Curse of the Black Pearl'): [['戈尔·韦宾斯基', 'Gore Verbinski'], ['约翰尼·德普', 'Johnny Depp', '杰弗里·拉什', 'Geoffrey Rush', '奥兰多·布鲁姆', 'Orlando Bloom']],
    ('惊魂记', 'Psycho'): [['阿尔弗雷德·希区柯克', 'Alfred Hitchcock'], ['安东尼·帕金斯', 'Anthony Perkins', '珍妮特·利', 'Janet Leigh', '维拉·迈尔斯', 'Vera Miles']],
    ('低俗小说', 'Pulp Fiction'): [['昆汀·塔伦蒂诺', 'Quentin Tarantino'], ['约翰·特拉沃尔塔', 'John Travolta', '塞缪尔·L·杰克逊', 'Samuel L. Jackson', '乌玛·瑟曼', 'Uma Thurman']],
    ('拯救大兵瑞恩', 'Saving Private Ryan'): [['史蒂文·斯皮尔伯格', 'Steven Spielberg'], ['汤姆·汉克斯', 'Tom Hanks', '马特·达蒙', 'Matt Damon', '汤姆·西泽摩尔', 'Tom Sizemore']],
    ('辛德勒的名单', "Schindler's List"): [['史蒂文·斯皮尔伯格', 'Steven Spielberg'], ['连姆·尼森', 'Liam Neeson', '本·金斯利', 'Ben Kingsley', '拉尔夫·费因斯', 'Ralph Fiennes']],
    ('七宗罪', 'Se7en'): [['大卫·芬奇', 'David Fincher'], ['摩根·弗里曼', 'Morgan Freeman', '布拉德·皮特', 'Brad Pitt', '格温妮丝·帕特洛', 'Gwyneth Paltrow']],
    ('禁闭岛', 'Shutter Island'): [['马丁·斯科塞斯', 'Martin Scorsese'], ['莱昂纳多·迪卡普里奥', 'Leonardo DiCaprio', '马克·鲁法洛', 'Mark Ruffalo', '本·金斯利', 'Ben Kingsley']],
    ('贫民窟的百万富翁', 'Slumdog Millionaire'): [['丹尼·博伊尔', 'Danny Boyle', '洛芙琳·坦丹', 'Loveleen Tandan'], ['戴夫·帕特尔', 'Dev Patel', '芙蕾达·平托', 'Freida Pinto', '马德胡尔·米塔尔', 'Madhur Mittal']],
    ('蜘蛛侠：平行宇宙', 'Spider-Man: Into the Spider-Verse'): [['鲍勃·佩尔西凯蒂', 'Bob Persichetti', '彼得·拉姆齐', 'Peter Ramsey', '罗德尼·罗斯曼', 'Rodney Rothman'], ['沙梅克·摩尔', 'Shameik Moore', '杰克·约翰逊', 'Jake Johnson', '海莉·斯坦菲尔德', 'Hailee Steinfeld']],
    ('千与千寻', 'Spirited Away'): [['宫崎骏', 'Hayao Miyazaki'], ['柊瑠美', 'Rumi Hiiragi', '入野自由', 'Miyu Irino', '夏木真理', 'Mari Natsuki']],
    ('终结者2：审判日', 'Terminator 2: Judgment Day'): [['詹姆斯·卡梅隆', 'James Cameron'], ['阿诺·施瓦辛格', 'Arnold Schwarzenegger', '琳达·汉密尔顿', 'Linda Hamilton', '爱德华·弗朗', 'Edward Furlong']],
    ('谍影重重3', 'The Bourne Ultimatum'): [['保罗·格林格拉斯', 'Paul Greengrass'], ['马特·达蒙', 'Matt Damon', '茱莉亚·斯蒂尔斯', 'Julia Stiles', '大卫·史崔森', 'David Strathairn']],
    ('本杰明·巴顿奇事', 'The Curious Case of Benjamin Button'): [['大卫·芬奇', 'David Fincher'], ['布拉德·皮特', 'Brad Pitt', '凯特·布兰切特', 'Cate Blanchett', '塔拉吉·P·汉森', 'Taraji P. Henson']],
    ('蝙蝠侠：黑暗骑士', 'The Dark Knight'): [['克里斯托弗·诺兰', 'Christopher Nolan'], ['克里斯蒂安·贝尔', 'Christian Bale', '希斯·莱杰', 'Heath Ledger', '艾伦·艾克哈特', 'Aaron Eckhart']],
    ('教父', 'The Godfather'): [['弗朗西斯·福特·科波拉', 'Francis Ford Coppola'], ['马龙·白兰度', 'Marlon Brando', '阿尔·帕西诺', 'Al Pacino', '詹姆斯·卡安', 'James Caan']],
    ('布达佩斯大饭店', 'The Grand Budapest Hotel'): [['韦斯·安德森', 'Wes Anderson'], ['拉尔夫·费因斯', 'Ralph Fiennes', 'F·默里·亚伯拉罕', 'F. Murray Abraham', '马修·阿马立克', 'Mathieu Amalric']],
    ('绿里奇迹', 'The Green Mile'): [['弗兰克·德拉邦特', 'Frank Darabont'], ['汤姆·汉克斯', 'Tom Hanks', '迈克尔·克拉克·邓肯', 'Michael Clarke Duncan', '大卫·摩斯', 'David Morse']],
    ('模仿游戏', 'The Imitation Game'): [['莫滕·泰杜姆', 'Morten Tyldum'], ['本尼迪克特·康伯巴奇', 'Benedict Cumberbatch', '凯拉·奈特莉', 'Keira Knightley', '马修·古德', 'Matthew Goode']],
    ('触不可及', 'The Intouchables'): [['奥利维埃·纳卡什', 'Olivier Nakache', '埃里克·托莱达诺', 'Éric Toledano'], ['弗朗索瓦·克鲁塞', 'François Cluzet', '奥马尔·赛', 'Omar Sy', '安妮·勒尼', 'Anne Le Ny']],
    ('狮子王', 'The Lion King'): [['罗杰·阿勒斯', 'Roger Allers', '罗伯·明科夫', 'Rob Minkoff'], ['马修·布罗德里克', 'Matthew Broderick', '杰瑞米·艾恩斯', 'Jeremy Irons', '詹姆斯·厄尔·琼斯', 'James Earl Jones']],
    ('指环王：王者归来', 'The Lord of the Rings: The Return of the King'): [['彼得·杰克逊', 'Peter Jackson'], ['伊利亚·伍德', 'Elijah Wood', '伊恩·麦克莱恩', 'Ian McKellen', '维果·莫滕森', 'Viggo Mortensen']],
    ('火星救援', 'The Martian'): [['雷德利·斯科特', 'Ridley Scott'], ['马特·达蒙', 'Matt Damon', '杰西卡·查斯坦', 'Jessica Chastain', '克里斯汀·韦格', 'Kristen Wiig']],
    ('黑客帝国', 'The Matrix'): [['拉娜·沃卓斯基', 'Lana Wachowski', '莉莉·沃卓斯基', 'Lilly Wachowski'], ['基努·里维斯', 'Keanu Reeves', '劳伦斯·菲什伯恩', 'Laurence Fishburne', '凯瑞-安·莫斯', 'Carrie-Anne Moss']],
    ('恋恋笔记本', 'The Notebook'): [['尼克·卡萨维茨', 'Nick Cassavetes'], ['瑞恩·高斯林', 'Ryan Gosling', '瑞秋·麦克亚当斯', 'Rachel McAdams', '詹姆斯·加纳', 'James Garner']],
    ('钢琴家', 'The Pianist'): [['罗曼·波兰斯基', 'Roman Polanski'], ['阿德里安·布劳迪', 'Adrien Brody', '托马斯·克雷茨曼', 'Thomas Kretschmann', '弗兰克·芬利', 'Frank Finlay']],
    ('致命魔术', 'The Prestige'): [['克里斯托弗·诺兰', 'Christopher Nolan'], ['休·杰克曼', 'Hugh Jackman', '克里斯蒂安·贝尔', 'Christian Bale', '迈克尔·凯恩', 'Michael Caine']],
    ('肖申克的救赎', 'The Shawshank Redemption'): [['弗兰克·德拉邦特', 'Frank Darabont'], ['蒂姆·罗宾斯', 'Tim Robbins', '摩根·弗里曼', 'Morgan Freeman', '鲍勃·冈顿', 'Bob Gunton']],
    ('沉默的羔羊', 'The Silence of the Lambs'): [['乔纳森·德米', 'Jonathan Demme'], ['朱迪·福斯特', 'Jodie Foster', '安东尼·霍普金斯', 'Anthony Hopkins', '斯科特·格伦', 'Scott Glenn']],
    ('第六感', 'The Sixth Sense'): [['M·奈特·沙马兰', 'M. Night Shyamalan'], ['布鲁斯·威利斯', 'Bruce Willis', '海利·乔·奥斯蒙', 'Haley Joel Osment', '托妮·科莱特', 'Toni Collette']],
    ('楚门的世界', 'The Truman Show'): [['彼得·威尔', 'Peter Weir'], ['金·凯瑞', 'Jim Carrey', '劳拉·琳妮', 'Laura Linney', '诺亚·埃默里奇', 'Noah Emmerich']],
    ('泰坦尼克号', 'Titanic'): [['詹姆斯·卡梅隆', 'James Cameron'], ['莱昂纳多·迪卡普里奥', 'Leonardo DiCaprio', '凯特·温斯莱特', 'Kate Winslet', '比利·赞恩', 'Billy Zane']],
    ('玩具总动员3', 'Toy Story 3'): [['李·昂克里奇', 'Lee Unkrich'], ['汤姆·汉克斯', 'Tom Hanks', '蒂姆·艾伦', 'Tim Allen', '琼·库萨克', 'Joan Cusack']],
    ('飞屋环游记', 'Up'): [['皮特·道格特', 'Pete Docter', '鲍勃·彼德森', 'Bob Peterson'], ['艾德·阿斯纳', 'Ed Asner', '克里斯托弗·普卢默', 'Christopher Plummer', '乔丹·长井', 'Jordan Nagai']],
    ('机器人总动员', 'WALL·E'): [['安德鲁·斯坦顿', 'Andrew Stanton'], ['本·伯特', 'Ben Burtt', '艾丽莎·奈特', 'Elissa Knight', '杰夫·加林', 'Jeff Garlin']],
    ('爆裂鼓手', 'Whiplash'): [['达米恩·查泽雷', 'Damien Chazelle'], ['迈尔斯·特勒', 'Miles Teller', 'J·K·西蒙斯', 'J.K. Simmons', '保罗·雷泽', 'Paul Reiser']]
}

In [142]:
# Load CSVs
df_cn = pd.read_csv('../douban_reviews.csv')

#load in Chinese stopwords
zh_stopwords = set(open('../text_processing/chinese_stopwords.txt', encoding='utf-8').read().split())

In [106]:
import re
import jieba
import string
from typing import List, Tuple, Set

# Special tokens
SPECIAL_TOKENS = {"<MOVIE_TITLE>", "<DIRECTOR_NAME>", "<ACTOR_NAME>"}

# Context indicators for Chinese
DIRECTOR_INDICATORS_ZH = {'导演', '执导', '指导', '拍摄', '拍'}
ACTOR_INDICATORS_ZH = {'主演', '饰演', '演员', '出演', '扮演', '演'}
NAME_INDICATORS_ZH = DIRECTOR_INDICATORS_ZH | ACTOR_INDICATORS_ZH

# Context indicators for English
DIRECTOR_INDICATORS_EN = {'directed', 'director', 'directed by', 'filmmaker'}
ACTOR_INDICATORS_EN = {'starring', 'actor', 'actress', 'played', 'stars', 'cast'}
NAME_INDICATORS_EN = DIRECTOR_INDICATORS_EN | ACTOR_INDICATORS_EN

# Common Chinese surnames that need context validation
COMMON_SURNAMES_ZH = {
    '李', '王', '张', '刘', '陈', '杨', '黄', '赵', '周', '吴', 
    '徐', '孙', '朱', '马', '胡', '郭', '林', '何', '高', '梁'
}

def segment_mixed_text(text):
    """Segment mixed Chinese-English text using jieba while preserving English words."""
    # Use jieba to segment
    return list(jieba.cut(text))

def normalize_for_matching(text):
    """Normalize text for matching - handles both Chinese and English."""
    # Remove Chinese and English punctuation for matching
    text = re.sub(r'[^\w\s\u4e00-\u9fff]', '', text)
    # Lowercase only ASCII characters
    return ''.join(c.lower() if ord(c) < 128 else c for c in text)

def find_title_mentions(text, title_tuple):
    """Find movie title mentions for both Chinese and English titles."""
    matches = []
    
    # Extract Chinese and English titles
    zh_title = title_tuple[0] if isinstance(title_tuple, tuple) else title_tuple
    en_title = title_tuple[1] if isinstance(title_tuple, tuple) and len(title_tuple) > 1 else None
    
    # Find Chinese title matches
    if zh_title and len(zh_title) >= 2:
        # Exact match
        pattern = re.escape(zh_title)
        for m in re.finditer(pattern, text):
            matches.append((m.start(), m.end(), "<MOVIE_TITLE>"))
        
        # Partial match for longer titles (e.g., "盗梦" for "盗梦空间")
        if len(zh_title) >= 4:
            partial = zh_title[:2]
            pattern = re.escape(partial)
            for m in re.finditer(pattern, text):
                # Check if it's not part of the full title already matched
                if not any(m.start() >= start and m.end() <= end for start, end, _ in matches):
                    # Validate context - avoid matching if it's part of another word
                    if is_valid_title_partial(text, m.start(), m.end(), partial):
                        matches.append((m.start(), m.end(), "<MOVIE_TITLE>"))
    
    # Find English title matches (case-insensitive)
    if en_title and len(en_title) >= 3:
        pattern = r'\b' + re.escape(en_title) + r'\b'
        for m in re.finditer(pattern, text, flags=re.IGNORECASE):
            matches.append((m.start(), m.end(), "<MOVIE_TITLE>"))
    
    return matches

def is_valid_title_partial(text, start, end, partial):
    """Validate if a partial title match is likely a real reference."""
    # Check surrounding context
    before_context = text[max(0, start-10):start]
    after_context = text[end:min(len(text), end+10)]
    
    # Positive indicators
    positive_patterns = ['看了', '看过', '这部', '那部', '电影', '片子', '影片']
    for pattern in positive_patterns:
        if pattern in before_context or pattern in after_context:
            return True
    
    # Check if followed by more characters that would make it a different word
    if end < len(text) and '\u4e00' <= text[end] <= '\u9fff':
        return False
    
    return True

def find_name_mentions(text, names, name_type):
    """Find director/actor name mentions with Chinese-English support."""
    matches = []
    
    for name_data in names:
        # Handle both string and list formats
        if isinstance(name_data, list):
            zh_name = name_data[0] if len(name_data) > 0 else None
            en_name = name_data[1] if len(name_data) > 1 else None
        else:
            zh_name = name_data
            en_name = None
        
        # Process Chinese name
        if zh_name:
            matches.extend(find_chinese_name_matches(text, zh_name, name_type))
        
        # Process English name
        if en_name:
            matches.extend(find_english_name_matches(text, en_name, name_type))
    
    return matches

def find_chinese_name_matches(text, name, name_type):
    """Find Chinese name matches with context validation."""
    matches = []
    
    # Full name match
    if len(name) >= 2:
        pattern = re.escape(name)
        for m in re.finditer(pattern, text):
            matches.append((m.start(), m.end(), name_type))
    
    # Surname matching with context
    if len(name) >= 2:
        surname = name[0]  # Chinese surnames come first
        
        # Only match single character surnames with strong context
        if surname in COMMON_SURNAMES_ZH:
            # Look for patterns like "李导演", "王主演"
            for indicator in NAME_INDICATORS_ZH:
                pattern = re.escape(surname) + indicator
                for m in re.finditer(pattern, text):
                    matches.append((m.start(), m.start() + 1, name_type))
        else:
            # Less common surnames can be matched more liberally
            pattern = re.escape(surname)
            for m in re.finditer(pattern, text):
                if is_likely_name_reference_zh(text, m.start(), m.end(), name_type):
                    matches.append((m.start(), m.end(), name_type))
    
    # Two-character surname matching (for names with 3+ characters)
    if len(name) >= 3:
        surname_2char = name[:2]
        pattern = re.escape(surname_2char)
        for m in re.finditer(pattern, text):
            if is_likely_name_reference_zh(text, m.start(), m.end(), name_type):
                matches.append((m.start(), m.end(), name_type))
    
    return matches

def find_english_name_matches(text, name, name_type):
    """Find English name matches similar to the original pipeline."""
    matches = []
    
    # Full name match (case-insensitive)
    pattern = r'\b' + re.escape(name) + r'\b'
    for m in re.finditer(pattern, text, flags=re.IGNORECASE):
        matches.append((m.start(), m.end(), name_type))
    
    # Last name only
    name_parts = name.split()
    if len(name_parts) >= 2:
        last_name = name_parts[-1]
        
        # Last name with possessive
        possessive_pattern = r'\b' + re.escape(last_name) + r"'s?\b"
        for m in re.finditer(possessive_pattern, text, flags=re.IGNORECASE):
            matches.append((m.start(), m.end(), name_type))
        
        # Last name with context
        if len(last_name) >= 4:  # Only for distinctive last names
            pattern = r'\b' + re.escape(last_name) + r'\b'
            for m in re.finditer(pattern, text, flags=re.IGNORECASE):
                if is_likely_name_reference_en(text, m.start(), m.end(), name_type):
                    matches.append((m.start(), m.end(), name_type))
    
    return matches

def is_likely_name_reference_zh(text, start, end, name_type):
    """Check if Chinese text segment is likely a name reference."""
    # Look at surrounding context
    context_start = max(0, start - 15)
    context_end = min(len(text), end + 15)
    context = text[context_start:context_end]
    
    # Check for indicators based on type
    if name_type == "<DIRECTOR_NAME>":
        indicators = DIRECTOR_INDICATORS_ZH
    else:
        indicators = ACTOR_INDICATORS_ZH
    
    for indicator in indicators:
        if indicator in context:
            return True
    
    # Check for general name indicators
    general_indicators = ['的', '由', '和', '与', '跟']
    for indicator in general_indicators:
        if indicator in context:
            return True
    
    return False

def is_likely_name_reference_en(text, start, end, name_type):
    """Check if English text segment is likely a name reference."""
    context_start = max(0, start - 20)
    context_end = min(len(text), end + 20)
    context = text[context_start:context_end].lower()
    
    # Check for indicators based on type
    if name_type == "<DIRECTOR_NAME>":
        indicators = DIRECTOR_INDICATORS_EN
    else:
        indicators = ACTOR_INDICATORS_EN
    
    for indicator in indicators:
        if indicator in context:
            return True
    
    return False

def remove_overlapping(matches):
    """Remove overlapping matches, keeping the longest ones."""
    if not matches:
        return []
    
    # Sort by start position, then by length (longest first)
    matches = sorted(matches, key=lambda x: (x[0], -(x[1]-x[0])))
    
    result = []
    last_end = -1
    
    for start, end, tag in matches:
        if start >= last_end:
            result.append((start, end, tag))
            last_end = end
    
    return result

def find_movie_tuple_by_english_title(english_title, movies):
    """
    Find the movie tuple key by English title.
    
    Args:
        english_title: English movie title string
        movies: Dictionary with tuple keys (Chinese, English)
    
    Returns:
        The tuple key if found, None otherwise
    """
    for movie_tuple in movies.keys():
        # Check if it's a tuple with English title
        if isinstance(movie_tuple, tuple) and len(movie_tuple) > 1:
            if movie_tuple[1] == english_title:
                return movie_tuple
        # Also handle case where key might be just English title
        elif movie_tuple == english_title:
            return movie_tuple
    return None

def preprocess_chinese_movie_review(text, movies, title, zh_stopwords):
    """
    Preprocess a Chinese movie review with entity recognition.
    
    Args:
        text: The review text (Chinese with possible English)
        movies: Dictionary with movie information
        title: The movie title (can be English string or tuple)
        zh_stopwords: Set of Chinese stopwords
    """
    # If title is just English string, find the corresponding tuple
    if isinstance(title, str):
        title_key = find_movie_tuple_by_english_title(title, movies)
    else:
        title_key = title
    
    # Handle case where movie not in dictionary
    if title_key is None or title_key not in movies:
        # Basic preprocessing: segment and remove stopwords
        tokens = segment_mixed_text(text.lower())
        cleaned = []
        for token in tokens:
            # Skip if it's a stopword or punctuation
            if token in zh_stopwords:
                continue
            if all(c in string.punctuation or c in '，。！？：；""''（）【】' for c in token):
                continue
            if token.strip():
                cleaned.append(token)
        return ' '.join(cleaned)
    
    # Get movie information using the tuple key
    directors, actors = movies[title_key]
    
    # Find all entity matches
    entity_matches = []
    
    # Find title mentions (use the tuple for both Chinese and English)
    entity_matches.extend(find_title_mentions(text, title_key))
    
    # Find director mentions
    entity_matches.extend(find_name_mentions(text, directors, "<DIRECTOR_NAME>"))
    
    # Find actor mentions
    entity_matches.extend(find_name_mentions(text, actors, "<ACTOR_NAME>"))
    
    # Remove overlapping matches
    entity_matches = remove_overlapping(entity_matches)
    
    # Replace entities in text (from end to start to preserve indices)
    new_text = text
    for start, end, tag in sorted(entity_matches, key=lambda x: -x[0]):
        new_text = new_text[:start] + f" {tag} " + new_text[end:]
    
    # Segment the text with jieba
    tokens = segment_mixed_text(new_text)
    
    # Clean and filter tokens
    cleaned = []
    for token in tokens:
        token_lower = token.lower()
        
        # Keep special tokens (preserve original case with brackets)
        if token.upper() in SPECIAL_TOKENS or token in SPECIAL_TOKENS:
            cleaned.append(token)
            continue
        
        # Skip stopwords
        if token in zh_stopwords:
            continue
        
        # Skip pure punctuation
        if all(c in string.punctuation or c in '，。！？：；""''（）【】《》' for c in token):
            continue
        
        # Skip empty tokens
        if not token.strip():
            continue
        
        # Add the token
        cleaned.append(token)
    
    return ' '.join(cleaned)

# Example usage with dataframe
def process_chinese_reviews(df_cn, movies, zh_stopwords):
    """
    Process Chinese movie reviews in a dataframe.
    
    Args:
        df_cn: DataFrame with 'review_text' and 'title' columns (title can be English only)
        movies: Movie information dictionary with tuple keys
        zh_stopwords: Set of Chinese stopwords
    
    Returns:
        DataFrame with added 'processed_comment' column
    """
    # Initialize jieba for better segmentation
    jieba.initialize()
    
    # Process each review
    df_cn['processed_comment'] = df_cn.apply(
        lambda row: preprocess_chinese_movie_review(
            row['review_text'], 
            movies, 
            row['title'],  # Can be English string
            zh_stopwords
        ),
        axis=1
    )
    
    return df_cn

In [144]:
# Chinese preprocessing
def clean_chinese(text):
    words = jieba.lcut(text)
    words = [word for word in words if word not in zh_stopwords]
    return ' '.join(words)


def clean_chinese(text):
    # Remove newline and carriage return characters
    text = text.replace('\n', '').replace('\r', '')

    # Lowercase English letters only
    text = re.sub(r'[A-Za-z]+', lambda m: m.group(0).lower(), text)

    # Tokenize using jieba
    words = jieba.lcut(text)

    # Remove stopwords
    words = [word for word in words if word not in zh_stopwords]

    return ' '.join(words)

df_cn['processed_comment'] = df_cn['review_text'].apply(clean_chinese)

In [145]:
df_cn

,title,review_text,stars,url,processed_comment
0,Bohemian Rhapsody,一句话，电影就是不及格的！\n内容不尊重事实，各种想当然，而且表述和模仿都太流于表面，幼稚而...,1,https://movie.douban.com/review/10052620/,一句 话 电影 不及格 内容 尊重事实 想当然 表述 模仿 太流于 表面 幼稚 肤浅 年代 ...
1,Bohemian Rhapsody,一个天才音乐艺人，傲慢无礼，骄狂自大的性格。逃避家庭的叛逆灵魂，却依托了爱好音乐的一群队友。...,1,https://movie.douban.com/review/10076604/,天才 音乐 艺人 傲慢无礼 骄狂 自大 性格 逃避 家庭 叛逆 灵魂 依托 爱好音乐 一群 ...
2,Bohemian Rhapsody,在坐的给这片子打五星的，有几个是那个年代出生的？听过皇后乐队现场表演的有几个？一部叙事风格的...,1,https://movie.douban.com/review/9985323/,坐 片子 五星 几个 年代 出生 听过 皇后 乐队 现场表演 几个 一部 叙事 风格 故事片...
3,Bohemian Rhapsody,一个路人的角度，我喜欢听音乐，但也没有深入了解过，这部电影音乐我个人欣赏不来，从影片中我也没...,1,https://movie.douban.com/review/10544688/,路人 角度 喜欢 听 音乐 没有 深入 了解 这部 电影 音乐 个人 欣赏 影片 没 感受 ...
4,Bohemian Rhapsody,说句实话，作为电影来说，拍的有点太差了。整个节奏，氛围营造的没有代入感。很多人说是因为对口型...,1,https://movie.douban.com/review/10516274/,说句实话 电影 拍 有点 太差 整个 节奏 氛围 营造 没有 代入 感 很多 是因为 对口型...
...,...,...,...,...,...
2688,Life Is Beautiful,忍受着意大利语的吵闹看完了，冲着前半段男主追女主的情节及军医的谜语给个二星，但是恕我直言的确...,2,https://movie.douban.com/review/8975354/,忍受着 意大利语 吵闹 完 冲着 前半段 男主 追 女主 情节 军医 谜语 二星 恕我直言 ...
2689,Life Is Beautiful,从电影主题来看，是要将小人物的乐观和历史灾难做对比，以此更加衬托出人们美好的心灵，同时痛斥战...,2,https://movie.douban.com/review/9282691/,电影 主题 来看 小人物 乐观 历史 灾难 以此 更加 衬托出 美好 心灵 痛斥 战争 罪恶...
2690,Life Is Beautiful,真相虽然残酷，但是确实生命的必经部分，即使对于一个孩子而已，\n也是如此。\n\n有时候，即...,2,https://movie.douban.com/review/3211968/,真相 残酷 确实 生命 必经 部分 孩子 有时候 善意 欺骗 未必 打造出 幸福 花朵 敢于...
2691,Life Is Beautiful,RT.\n看拍摄我以为是1979年的片子呢……\n演技很差\n道具很假\n至于广泛渲染的煽情...,2,https://movie.douban.com/review/1407755/,rt . 拍摄 1979 片子 … … 演技 很差 道具 很假 广泛 渲染 煽情 ~ 约 舒...


In [146]:
#load in custom stopwords
custom_stopwords = set(open('../text_processing/custom_chinese_stopwords.txt', encoding='utf-8').read().split())
words = []
for word in custom_stopwords:
    word = word.lower()
    words.append(word)
custom_stopwords = set(words)

In [147]:
custom_stopwords

{'12',
 '12怒汉',
 '2001:',
 '2001太空漫游',
 '2:',
 '3',
 '<movie_title>',
 'a',
 'and',
 'angry',
 'avatar',
 'away',
 'barrels',
 'basterds',
 'beautiful',
 'benjamin',
 'black',
 'bohemian',
 'book',
 'bourne',
 'budapest',
 'button',
 'can',
 'caribbean:',
 'case',
 'catch',
 'club',
 'coco',
 "cuckoo's",
 'curious',
 'curse',
 'dark',
 'day',
 'django',
 'dragon',
 'fiction',
 'fight',
 'film',
 'films',
 'flew',
 'forrest',
 'frozen',
 'fury',
 'game',
 'girl',
 'godfather',
 'gone',
 'good',
 'grand',
 'green',
 'gump',
 'hacksaw',
 'harry',
 'hotel',
 'how',
 'hunting',
 'if',
 'imitation',
 'inc.',
 'inception',
 'inglourious',
 'inside',
 'interstellar',
 'into',
 'intouchables',
 'is',
 'island',
 'joker',
 'judgment',
 'just',
 'king',
 'knight',
 'know',
 'la',
 'lambs',
 'land',
 'life',
 'like',
 'lion',
 'list',
 'lock,',
 'lord',
 'léon:',
 'mad',
 'make',
 'martian',
 'matrix',
 'max:',
 'me',
 'memento',
 'men',
 'mile',
 'millionaire',
 'mind',
 'monsters,',
 'movie',
 '

In [148]:
df_cn

,title,review_text,stars,url,processed_comment
0,Bohemian Rhapsody,一句话，电影就是不及格的！\n内容不尊重事实，各种想当然，而且表述和模仿都太流于表面，幼稚而...,1,https://movie.douban.com/review/10052620/,一句 话 电影 不及格 内容 尊重事实 想当然 表述 模仿 太流于 表面 幼稚 肤浅 年代 ...
1,Bohemian Rhapsody,一个天才音乐艺人，傲慢无礼，骄狂自大的性格。逃避家庭的叛逆灵魂，却依托了爱好音乐的一群队友。...,1,https://movie.douban.com/review/10076604/,天才 音乐 艺人 傲慢无礼 骄狂 自大 性格 逃避 家庭 叛逆 灵魂 依托 爱好音乐 一群 ...
2,Bohemian Rhapsody,在坐的给这片子打五星的，有几个是那个年代出生的？听过皇后乐队现场表演的有几个？一部叙事风格的...,1,https://movie.douban.com/review/9985323/,坐 片子 五星 几个 年代 出生 听过 皇后 乐队 现场表演 几个 一部 叙事 风格 故事片...
3,Bohemian Rhapsody,一个路人的角度，我喜欢听音乐，但也没有深入了解过，这部电影音乐我个人欣赏不来，从影片中我也没...,1,https://movie.douban.com/review/10544688/,路人 角度 喜欢 听 音乐 没有 深入 了解 这部 电影 音乐 个人 欣赏 影片 没 感受 ...
4,Bohemian Rhapsody,说句实话，作为电影来说，拍的有点太差了。整个节奏，氛围营造的没有代入感。很多人说是因为对口型...,1,https://movie.douban.com/review/10516274/,说句实话 电影 拍 有点 太差 整个 节奏 氛围 营造 没有 代入 感 很多 是因为 对口型...
...,...,...,...,...,...
2688,Life Is Beautiful,忍受着意大利语的吵闹看完了，冲着前半段男主追女主的情节及军医的谜语给个二星，但是恕我直言的确...,2,https://movie.douban.com/review/8975354/,忍受着 意大利语 吵闹 完 冲着 前半段 男主 追 女主 情节 军医 谜语 二星 恕我直言 ...
2689,Life Is Beautiful,从电影主题来看，是要将小人物的乐观和历史灾难做对比，以此更加衬托出人们美好的心灵，同时痛斥战...,2,https://movie.douban.com/review/9282691/,电影 主题 来看 小人物 乐观 历史 灾难 以此 更加 衬托出 美好 心灵 痛斥 战争 罪恶...
2690,Life Is Beautiful,真相虽然残酷，但是确实生命的必经部分，即使对于一个孩子而已，\n也是如此。\n\n有时候，即...,2,https://movie.douban.com/review/3211968/,真相 残酷 确实 生命 必经 部分 孩子 有时候 善意 欺骗 未必 打造出 幸福 花朵 敢于...
2691,Life Is Beautiful,RT.\n看拍摄我以为是1979年的片子呢……\n演技很差\n道具很假\n至于广泛渲染的煽情...,2,https://movie.douban.com/review/1407755/,rt . 拍摄 1979 片子 … … 演技 很差 道具 很假 广泛 渲染 煽情 ~ 约 舒...


In [149]:
# English preprocessing
def clean_english(text):
    text = str(text).lower()
    words = [w for w in text.split() if w not in custom_stopwords]
    return ' '.join(words)

# Apply preprocessing
df_cn['clean_comment'] = df_cn['processed_comment'].apply(clean_english)

In [150]:
df_cn

,title,review_text,stars,url,processed_comment,clean_comment
0,Bohemian Rhapsody,一句话，电影就是不及格的！\n内容不尊重事实，各种想当然，而且表述和模仿都太流于表面，幼稚而...,1,https://movie.douban.com/review/10052620/,一句 话 电影 不及格 内容 尊重事实 想当然 表述 模仿 太流于 表面 幼稚 肤浅 年代 ...,一句 话 电影 不及格 内容 尊重事实 想当然 表述 模仿 太流于 表面 幼稚 肤浅 年代 ...
1,Bohemian Rhapsody,一个天才音乐艺人，傲慢无礼，骄狂自大的性格。逃避家庭的叛逆灵魂，却依托了爱好音乐的一群队友。...,1,https://movie.douban.com/review/10076604/,天才 音乐 艺人 傲慢无礼 骄狂 自大 性格 逃避 家庭 叛逆 灵魂 依托 爱好音乐 一群 ...,天才 音乐 艺人 傲慢无礼 骄狂 自大 性格 逃避 家庭 叛逆 灵魂 依托 爱好音乐 一群 ...
2,Bohemian Rhapsody,在坐的给这片子打五星的，有几个是那个年代出生的？听过皇后乐队现场表演的有几个？一部叙事风格的...,1,https://movie.douban.com/review/9985323/,坐 片子 五星 几个 年代 出生 听过 皇后 乐队 现场表演 几个 一部 叙事 风格 故事片...,坐 片子 五星 几个 年代 出生 听过 皇后 乐队 现场表演 几个 一部 叙事 风格 故事片...
3,Bohemian Rhapsody,一个路人的角度，我喜欢听音乐，但也没有深入了解过，这部电影音乐我个人欣赏不来，从影片中我也没...,1,https://movie.douban.com/review/10544688/,路人 角度 喜欢 听 音乐 没有 深入 了解 这部 电影 音乐 个人 欣赏 影片 没 感受 ...,路人 角度 喜欢 听 音乐 没有 深入 了解 这部 电影 音乐 个人 欣赏 影片 没 感受 ...
4,Bohemian Rhapsody,说句实话，作为电影来说，拍的有点太差了。整个节奏，氛围营造的没有代入感。很多人说是因为对口型...,1,https://movie.douban.com/review/10516274/,说句实话 电影 拍 有点 太差 整个 节奏 氛围 营造 没有 代入 感 很多 是因为 对口型...,说句实话 电影 拍 有点 太差 整个 节奏 氛围 营造 没有 代入 感 很多 是因为 对口型...
...,...,...,...,...,...,...
2688,Life Is Beautiful,忍受着意大利语的吵闹看完了，冲着前半段男主追女主的情节及军医的谜语给个二星，但是恕我直言的确...,2,https://movie.douban.com/review/8975354/,忍受着 意大利语 吵闹 完 冲着 前半段 男主 追 女主 情节 军医 谜语 二星 恕我直言 ...,忍受着 意大利语 吵闹 完 冲着 前半段 男主 追 女主 情节 军医 谜语 二星 恕我直言 ...
2689,Life Is Beautiful,从电影主题来看，是要将小人物的乐观和历史灾难做对比，以此更加衬托出人们美好的心灵，同时痛斥战...,2,https://movie.douban.com/review/9282691/,电影 主题 来看 小人物 乐观 历史 灾难 以此 更加 衬托出 美好 心灵 痛斥 战争 罪恶...,电影 主题 来看 小人物 乐观 历史 灾难 以此 更加 衬托出 美好 心灵 痛斥 战争 罪恶...
2690,Life Is Beautiful,真相虽然残酷，但是确实生命的必经部分，即使对于一个孩子而已，\n也是如此。\n\n有时候，即...,2,https://movie.douban.com/review/3211968/,真相 残酷 确实 生命 必经 部分 孩子 有时候 善意 欺骗 未必 打造出 幸福 花朵 敢于...,真相 残酷 确实 生命 必经 部分 孩子 有时候 善意 欺骗 未必 打造出 幸福 花朵 敢于...
2691,Life Is Beautiful,RT.\n看拍摄我以为是1979年的片子呢……\n演技很差\n道具很假\n至于广泛渲染的煽情...,2,https://movie.douban.com/review/1407755/,rt . 拍摄 1979 片子 … … 演技 很差 道具 很假 广泛 渲染 煽情 ~ 约 舒...,rt . 拍摄 1979 片子 … … 演技 很差 道具 很假 广泛 渲染 煽情 ~ 约 舒...


In [151]:
comments = df_cn["clean_comment"].dropna().tolist()

In [153]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-mpnet-base-v2")

embeddings = model.encode(comments, show_progress_bar=True)

/Users/elimarx/miniconda3/lib/python3.13/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange
/Users/elimarx/miniconda3/lib/python3.13/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/Users/elimarx/miniconda3/lib/python3.13/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/Users/elimarx/miniconda3/lib/python3.13/site-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0

In [154]:
#do clustering
from sklearn.cluster import KMeans

num_clusters = 6 # tweak this
kmeans = KMeans(n_clusters=num_clusters, random_state=42)
labels = kmeans.fit_predict(embeddings)

df_cn['cluster'] = labels

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [155]:
df_cn['cluster'].value_counts()

cluster
1    764
2    441
3    433
4    394
5    386
0    275
Name: count, dtype: int64

In [156]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
from keybert import KeyBERT
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import seaborn as sns

def extract_keybert_keywords(df_en, model, num_keywords=10):
    print("=== KEYBERT KEYWORDS ===\n")
    kw_model = KeyBERT(model)
    for i in range(num_clusters):
        cluster_comments = df_en[df_en['cluster'] == i]['clean_comment'].tolist()
        joined = " ".join(cluster_comments)
        keywords = kw_model.extract_keywords(joined, top_n=num_keywords)
        print(f"Cluster {i} keywords:", keywords)

def analyze_clusters_comprehensive(df_en, text_column='processed_comment'):
    """
    Comprehensive cluster analysis with multiple methods
    """
    
    # 1. TF-IDF Based Analysis (most discriminative terms per cluster)
    print("=== TF-IDF DISCRIMINATIVE TERMS ===\n")
    
    # Create a document for each cluster by joining all comments
    cluster_docs = []
    for i in range(df_en['cluster'].nunique()):
        cluster_text = ' '.join(df_en[df_en['cluster'] == i][text_column].tolist())
        cluster_docs.append(cluster_text)
    
    # Compute TF-IDF
    tfidf = TfidfVectorizer(max_features=100, ngram_range=(1, 3), stop_words='english')
    tfidf_matrix = tfidf.fit_transform(cluster_docs)
    feature_names = tfidf.get_feature_names_out()
    
    # Get top terms for each cluster based on TF-IDF scores
    for i in range(len(cluster_docs)):
        scores = tfidf_matrix[i].toarray().flatten()
        top_indices = scores.argsort()[-15:][::-1]  # Top 15 terms
        top_terms = [(feature_names[idx], scores[idx]) for idx in top_indices]
        
        print(f"Cluster {i} - Most Discriminative Terms:")
        for term, score in top_terms[:10]:
            print(f"  {term}: {score:.3f}")
        print()
    
    # 3. Movie and Rating Distribution
    print("\n=== MOVIE & RATING DISTRIBUTION ===\n")
    
    for i in range(df_en['cluster'].nunique()):
        cluster_df = df_en[df_en['cluster'] == i]
        
        print(f"Cluster {i}:")
        print(f"  Size: {len(cluster_df)} reviews")
        
        # Top movies in this cluster
        top_movies = cluster_df['title'].value_counts().head(5)
        print(f"  Top movies:")
        for movie, count in top_movies.items():
            print(f"    - {movie}: {count} reviews")
        
        # Rating distribution if available
        if 'rating' in df_en.columns:
            print(f"  Average rating: {cluster_df['rating'].mean():.2f}")
            print(f"  Rating std: {cluster_df['rating'].std():.2f}")
        
        print()

    # 5. Length Analysis
    print("\n=== REVIEW LENGTH ANALYSIS ===\n")
    
    df_en['review_length'] = df_en[text_column].str.split().str.len()
    
    for i in range(df_en['cluster'].nunique()):
        cluster_lengths = df_en[df_en['cluster'] == i]['review_length']
        print(f"Cluster {i}:")
        print(f"  Average length: {cluster_lengths.mean():.1f} words")
        print(f"  Median length: {cluster_lengths.median():.1f} words")
        print()
    
    # 4. Special Token Analysis
    print("\n=== SPECIAL TOKEN USAGE ===\n")
    
    special_tokens = [ 'director_name', 'actor_name']
    
    for i in range(df_en['cluster'].nunique()):
        cluster_comments = df_en[df_en['cluster'] == i][text_column].tolist()
        all_text = ' '.join(cluster_comments)
        
        print(f"Cluster {i}:")
        for token in special_tokens:
            count = all_text.count(token)
            avg_per_review = count / len(cluster_comments)
            print(f"  {token}: {count} total ({avg_per_review:.2f} per review)")
        print()

def get_sample_reviews(df_en, n_samples=3):
#print some random comments from each cluster
    """
    Get representative reviews from each cluster
    """
    print("\n=== RANDOM SAMPLE REVIEWS ===\n")
    
    for cluster_num in sorted(df_en['cluster'].unique()):
        print(f"\nCluster {cluster_num}:")
        sample_comments = df_en[df_en['cluster'] == cluster_num]['review_text'].sample(n_samples, random_state=42)
        for comment in sample_comments:
            print(" -", comment)
            print()
        print("-" * 40)

def extract_unique_ngrams(df_en, text_column='processed_comment'):
    """
    Find n-grams that are unique or highly specific to each cluster
    """
    print("\n=== UNIQUE N-GRAMS PER CLUSTER ===\n")
    
    from sklearn.feature_extraction.text import CountVectorizer
    
    # Get n-grams for each cluster
    cluster_ngrams = {}
    
    for i in range(df_en['cluster'].nunique()):
        cluster_text = df_en[df_en['cluster'] == i][text_column].tolist()
        
        # Extract bigrams and trigrams
        vectorizer = CountVectorizer(ngram_range=(2, 3), max_features=100)
        try:
            ngram_counts = vectorizer.fit_transform(cluster_text)
            feature_names = vectorizer.get_feature_names_out()
            
            # Sum occurrences across all documents in cluster
            total_counts = ngram_counts.sum(axis=0).A1
            ngram_freq = list(zip(feature_names, total_counts))
            ngram_freq.sort(key=lambda x: x[1], reverse=True)
            
            cluster_ngrams[i] = dict(ngram_freq[:20])  # Top 20 n-grams
        except:
            cluster_ngrams[i] = {}
    
    # Find n-grams unique to each cluster
    for i in range(df_en['cluster'].nunique()):
        print(f"Cluster {i} - Unique/Distinctive N-grams:")
        
        current_ngrams = set(cluster_ngrams[i].keys())
        other_ngrams = set()
        
        for j in range(df_en['cluster'].nunique()):
            if j != i:
                other_ngrams.update(cluster_ngrams[j].keys())
        
        unique_ngrams = current_ngrams - other_ngrams
        
        if unique_ngrams:
            for ngram in list(unique_ngrams)[:10]:
                print(f"  - {ngram}: {cluster_ngrams[i][ngram]} occurrences")
        else:
            # If no unique n-grams, show most frequent
            for ngram, count in list(cluster_ngrams[i].items())[:5]:
                print(f"  - {ngram}: {count} occurrences")
        print()

def visualize_clusters(df_en, embeddings):
    """
    Visualize clusters using dimensionality reduction
    """
    # PCA visualization
    pca = PCA(n_components=2)
    pca_coords = pca.fit_transform(embeddings)
    
    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 1)
    scatter = plt.scatter(pca_coords[:, 0], pca_coords[:, 1], 
                         c=df_en['cluster'], cmap='tab10', alpha=0.6)
    plt.colorbar(scatter)
    plt.title('Clusters in PCA Space')
    plt.xlabel('PC1')
    plt.ylabel('PC2')
    
    # t-SNE visualization (for smaller datasets)
    if len(df_en) < 5000:  # t-SNE is computationally expensive
        tsne = TSNE(n_components=2, random_state=42)
        tsne_coords = tsne.fit_transform(embeddings)
        
        plt.subplot(1, 2, 2)
        scatter = plt.scatter(tsne_coords[:, 0], tsne_coords[:, 1], 
                             c=df_en['cluster'], cmap='tab10', alpha=0.6)
        plt.colorbar(scatter)
        plt.title('Clusters in t-SNE Space')
        plt.xlabel('t-SNE 1')
        plt.ylabel('t-SNE 2')
    
    plt.tight_layout()
    plt.show()

def visualize_clusters2(df_en, embeddings):
    """
    Visualize clusters using dimensionality reduction but different output
    """
    pca = PCA(n_components=2)
    reduced = pca.fit_transform(embeddings)

    df_en['x'] = reduced[:, 0]
    df_en['y'] = reduced[:, 1]

    plt.figure(figsize=(10, 7))
    sns.scatterplot(data=df_en, x='x', y='y', hue='cluster', palette='tab10')
    plt.title("IMDB Comment Clusters")
    plt.show()

def analyze_cluster_themes(df_en, text_column='processed_comment'):
    """
    High-level theme analysis based on semantic patterns
    """
    print("\n=== POTENTIAL CLUSTER THEMES ===\n")
    
    # Define theme indicators
    theme_patterns = {
        'Technical/Cinematography': ['cinematography', 'camera', 'shot', 'visual', 'effects', 'lighting', 'score', 'soundtrack'],
        'Plot/Story': ['plot', 'story', 'narrative', 'ending', 'twist', 'predictable', 'boring', 'confusing'],
        'Acting/Performance': ['<ACTOR_NAME>', 'actor_name', 'acting', 'performance', 'character', 'role', 'cast'],
        'Direction': ['<DIRECTOR_NAME>', 'director_name', 'direction', 'directed', 'filmmaker'],
        'Genre-specific': ['horror', 'comedy', 'thriller', 'drama', 'action', 'romantic'],
        'Emotional Response': ['love', 'hate', 'disappoint', 'amazing', 'terrible', 'worst', 'best', 'masterpiece'],
        'Critical/Analytical': ['overrated', 'underrated', 'critic', 'review', 'analysis', 'interpret'],
    }
    
    for i in range(df_en['cluster'].nunique()):
        cluster_text = ' '.join(df_en[df_en['cluster'] == i][text_column].tolist()).lower()
        
        print(f"\nCluster {i} - Theme Scores:")
        theme_scores = {}
        
        for theme, keywords in theme_patterns.items():
            score = sum(cluster_text.count(keyword.lower()) for keyword in keywords)
            theme_scores[theme] = score
        
        # Normalize by cluster size
        cluster_size = len(df_en[df_en['cluster'] == i])
        
        # Sort themes by score
        sorted_themes = sorted(theme_scores.items(), key=lambda x: x[1], reverse=True)
        
        for theme, score in sorted_themes[:5]:
            normalized_score = score / cluster_size
            print(f"  {theme}: {normalized_score:.2f} mentions per review")


In [157]:
extract_keybert_keywords(df_cn, model, num_keywords=10)

=== KEYBERT KEYWORDS ===

Cluster 0 keywords: [('曲线救国', 0.5779), ('雨中曲', 0.5728), ('安魂曲', 0.5373), ('曲高和寡', 0.5354), ('堂而皇之', 0.533), ('狂想曲', 0.529), ('交响曲', 0.529), ('片尾曲', 0.529), ('钢琴曲', 0.529), ('主题曲', 0.527)]
Cluster 1 keywords: [('冠冕堂皇', 0.6257), ('富丽堂皇', 0.6257), ('仓皇', 0.5899), ('教皇', 0.5899), ('沙皇', 0.5899), ('堂而皇之', 0.5861), ('堂皇', 0.5805), ('皇上', 0.5712), ('新古典主义', 0.5594), ('齐家治国平天下', 0.5531)]
Cluster 2 keywords: [('平分秋色', 0.5258), ('白雪公主', 0.5234), ('大肆宣扬', 0.5195), ('英雄主义', 0.519), ('对立面', 0.5133), ('之下', 0.5119), ('清新', 0.5093), ('相比之下', 0.5013), ('标新立异', 0.5004), ('下里巴人', 0.4945)]
Cluster 3 keywords: [('男主不竖持', 0.61), ('男主不离', 0.6075), ('男主不敌', 0.6075), ('男主不穷', 0.6075), ('男主会', 0.5795), ('男主瑞恩', 0.5718), ('男主拉着', 0.5718), ('男主共情', 0.5718), ('男主由頭', 0.5718), ('男主不信', 0.5688)]
Cluster 4 keywords: [('高水平', 0.5113), ('大男子主义', 0.5022), ('伪文青', 0.4995), ('逼文青', 0.4995), ('老文青', 0.4995), ('高清晰', 0.4961), ('有水', 0.4955), ('不清', 0.4908), ('有多水', 0.4907), ('技术水平', 0.4907)]
Clust

In [159]:
# Run comprehensive analysis
analyze_clusters_comprehensive(df_cn, text_column='clean_comment')

# Extract unique n-grams
extract_unique_ngrams(df_cn, text_column='processed_comment')

# Analyze themes
analyze_cluster_themes(df_cn, text_column='processed_comment')

#get random reviews
get_sample_reviews(df_cn, n_samples=3)

# If you have embeddings available
visualize_clusters(df_cn, embeddings)
visualize_clusters2(df_cn, embeddings)

=== TF-IDF DISCRIMINATIVE TERMS ===

Cluster 0 - Most Discriminative Terms:
  电影: 0.578
  没有: 0.356
  故事: 0.181
  最后: 0.156
  这部: 0.155
  觉得: 0.151
  音乐: 0.148
  梦想: 0.145
  喜欢: 0.144
  真的: 0.142

Cluster 1 - Most Discriminative Terms:
  电影: 0.630
  没有: 0.339
  这部: 0.187
  故事: 0.159
  影片: 0.150
  这种: 0.133
  导演: 0.133
  觉得: 0.131
  最后: 0.124
  这部 电影: 0.119

Cluster 2 - Most Discriminative Terms:
  电影: 0.664
  没有: 0.282
  知道: 0.250
  剧情: 0.193
  真的: 0.190
  这部: 0.177
  觉得: 0.160
  片子: 0.145
  导演: 0.137
  感觉: 0.128

Cluster 3 - Most Discriminative Terms:
  男主: 0.458
  电影: 0.413
  没有: 0.321
  女主: 0.300
  最后: 0.174
  觉得: 0.157
  这部: 0.148
  知道: 0.127
  故事: 0.125
  剧情: 0.124

Cluster 4 - Most Discriminative Terms:
  电影: 0.597
  没有: 0.315
  人类: 0.175
  故事: 0.175
  这部: 0.170
  导演: 0.163
  最后: 0.154
  觉得: 0.137
  地球: 0.135
  影片: 0.131

Cluster 5 - Most Discriminative Terms:
  电影: 0.547
  没有: 0.397
  觉得: 0.182
  这部: 0.168
  最后: 0.167
  真的: 0.157
  世界: 0.140
  看到: 0.137
  这种: 0.136
  故事: 0.124



KeyError: 'comment'

In [128]:
def print_cluster_table(df):
    # Build the pivot table
    table = df[['title', 'cluster']].value_counts().unstack(fill_value=0)
  
    # Convert floats to ints if needed
    table = table.astype(int)

    # Print each row cleanly
    print("Title".ljust(40), end="")
    for col in table.columns:
        print(f"Cluster {col}".rjust(12), end="")
    print()

    print("-" * (40 + 12 * len(table.columns)))
    
    for title, row in table.iterrows():
        print(title.ljust(40), end="")
        for count in row:
            print(str(count).rjust(12), end="")
        print()

In [129]:
print_cluster_table(df_en)

Title                                      Cluster 0   Cluster 1   Cluster 2   Cluster 3   Cluster 4   Cluster 5   Cluster 6   Cluster 7   Cluster 8   Cluster 9
----------------------------------------------------------------------------------------------------------------------------------------------------------------
12 Angry Men                                       4           2           0           0           4           3           5           0           3           4
2001: A Space Odyssey                              3          36           0          14          12          38          18           2          32          45
A Beautiful Mind                                  33           5           1           2          11           2          20           0           2          19
Avatar                                             3          27           4          12           9          13          10          10          47          64
Black Swan                        

In [130]:
table = df_en[['title', 'cluster']].value_counts().unstack()
table.loc